In [68]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep
from scipy.stats import norm
import warnings

# Configurações de exibição e avisos
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
warnings.filterwarnings('ignore')


In [69]:
ativos = ['ABEV3', 'B3SA3', 'BBAS3', 'BBSE3', 'EGIE3', 'FLRY3',
          'HYPE3', 'ITSA4', 'KLBN11', 'LEVE3', 'PETR4', 'TAEE11',
          'UNIP6', 'VALE3', 'RADL3']

In [70]:
# ---------------------------------------------------
# 2. DOWNLOAD E ATUALIZAÇÃO (CORREÇÃO JSON)
# ---------------------------------------------------
if not os.path.exists("dbJson"):
    os.makedirs("dbJson")

def atualizarDados(ativo):
    url = f"https://storage.googleapis.com/api-cdn-eaglesystem/api/{ativo.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return f"{ativo}: erro HTTP {response.status_code}"
        
        # Validação de conteúdo JSON para evitar erro 'Expecting value'
        content = response.text.strip()
        if not (content.startswith('{') or content.startswith('[')):
            return f"{ativo}: Erro - Conteúdo não é JSON válido"

        dados = response.json()
        with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
            json.dump(dados, arq, indent=2)
        return f"{ativo}: atualizado"
    except Exception as e:
        return f"{ativo}: erro -> {e}"

for ativo in ativos:
    print(atualizarDados(ativo))


ABEV3: atualizado
B3SA3: atualizado
BBAS3: atualizado
BBSE3: atualizado
EGIE3: atualizado
FLRY3: atualizado
HYPE3: atualizado
ITSA4: atualizado
KLBN11: atualizado
LEVE3: atualizado
PETR4: atualizado
TAEE11: atualizado
UNIP6: atualizado
VALE3: atualizado
RADL3: atualizado


In [71]:
# ---------------------------------------------------
# 3. PROCESSAMENTO DOS DADOS (DATAFRAME)
# ---------------------------------------------------
def dataFrameUnico(ativo):
    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]
    linhas = []

    for serie in dados["series"]:
        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:
            strike_price = strike["strike"]
            for tipo in ["call", "put"]:
                opt = strike[tipo]
                if opt:
  linhas.append({
                        "ativo": ativo, 
                        "symbol": opt["symbol"],
                        "categoria": tipo.upper(), 
                        "preco_atual": preco_atual,
                        "strike": strike_price, 
                        "bid": opt["bid"], 
                        "ask": opt["ask"], 
                        "volume": opt["volume"],
                        "maturity_type": opt["maturity_type"],
                        "moneyness": opt["bs"]["moneyness"],
                        "price": opt["bs"]["price"],
                        "delta": opt["bs"]["delta"], 
                        "gamma": opt["bs"]["gamma"], 
                        "vega": opt["bs"]["vega"],
                        "theta": opt["bs"]["theta"],
                        "rho": opt["bs"]["rho"],
                        "vol": opt["bs"]["volatility"], 
                        "poe": opt["bs"]["poe"],
                        "dias": dias,
                        "vencimento": vencimento,
                    })
    return pd.DataFrame(linhas)

todos = []
for ativo in ativos:

    df = dataFrameUnico(ativo)
    todos.append(df)


df_final = pd.concat(todos, ignore_index=True)

IndentationError: unindent does not match any outer indentation level (<string>, line 20)

In [ ]:
df_final

,ativo,symbol,categoria,preco_atual,strike,bid,ask,volume,maturity_type,moneyness,price,delta,gamma,vega,theta,rho,vol,poe,dias,vencimento
0,ABEV3,ABEVG960W2,CALL,15.87,9.56,0.00,0.0,0,AMERICAN,ITM,6.3326,1.000000,0.000000,0.000000,-0.005639,0.001514,0.000,100.00,4,2026-07-10
1,ABEV3,ABEVS960W2,PUT,15.87,9.56,0.00,0.0,0,EUROPEAN,OTM,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.00,4,2026-07-10
2,ABEV3,ABEVG980W2,CALL,15.87,9.76,0.00,0.0,0,AMERICAN,ITM,6.1331,1.000000,0.000000,0.000000,-0.005757,0.001546,0.000,100.00,4,2026-07-10
3,ABEV3,ABEVS980W2,PUT,15.87,9.76,0.00,0.0,0,EUROPEAN,OTM,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.00,4,2026-07-10
4,ABEV3,ABEVG100W2,CALL,15.87,9.96,0.00,0.0,0,AMERICAN,ITM,5.9335,1.000000,0.000000,0.000000,-0.005875,0.001577,0.000,100.00,4,2026-07-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21767,RADL3,RADLT200,PUT,16.78,19.29,0.00,0.0,0,EUROPEAN,ITM,2.5076,-0.390834,0.054769,0.068388,0.000622,-0.102170,33.406,55.59,284,2027-08-20
21768,RADL3,RADLL220,CALL,16.78,21.83,0.35,0.0,200,AMERICAN,OTM,2.8264,0.554463,0.049734,0.079813,-0.008132,0.093821,32.902,36.82,365,2027-12-17
21769,RADL3,RADLX220,PUT,16.78,21.83,0.00,0.0,0,EUROPEAN,ITM,3.6389,-0.445537,0.049734,0.079813,0.002270,-0.160991,43.258,63.18,365,2027-12-17
21770,RADL3,RADLL250,CALL,16.78,24.83,0.00,0.0,0,AMERICAN,OTM,2.0582,0.446322,0.049748,0.079835,-0.007515,0.078664,32.285,27.14,365,2027-12-17


In [ ]:
puts = df_final[df_final["categoria"] == "PUT"].copy()


puts["retorno"] = (puts["bid"] / (puts["strike"] - puts['bid'])) * 100
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"].replace(0, 1))
puts["dist_strike"] = ((puts["strike"] / puts["preco_atual"]) - 1) * 100


# Ranking e Score
filtro_put = puts[

    (puts['dias'].between(15, np.inf)) &
    (puts['dist_strike'] <= 0) &
    (puts['retorno_mes'] >= 0.95)


].copy()

if not filtro_put.empty:
    filtro_put['score'] = (
        filtro_put['dist_strike'].rank(ascending=True) * 1 +
        filtro_put['retorno_mes'].rank(ascending=False) * 4 +
        filtro_put['retorno'].rank(ascending=False) * 3
    )
    print("\n--- MELHORES OPÇÕES DE VENDA DE PUT ---")

    display(filtro_put[

        [
            'ativo', 'symbol', 'categoria','moneyness' , 'strike', 'preco_atual',
            'dist_strike', 'bid', 'ask',
            'volume', 'delta', 'theta', 'vol', 'poe',  'retorno', 'retorno_mes',
            'dias', 'vencimento', 'score',
        ]

    ].sort_values('score').head(30))


--- MELHORES OPÇÕES DE VENDA DE PUT ---


,ativo,symbol,categoria,moneyness,strike,preco_atual,dist_strike,bid,ask,volume,delta,theta,vol,poe,retorno,retorno_mes,dias,vencimento,score
21281,RADL3,RADLT165,PUT,OTM,16.43,16.78,-2.085816,0.68,0.70,27500,-0.360475,-0.009391,38.185,41.58,4.317460,3.809524,34,2026-08-21,103.0
21279,RADL3,RADLT162,PUT,OTM,16.18,16.78,-3.575685,0.58,0.61,400,-0.321585,-0.009265,39.351,37.50,3.717949,3.280543,34,2026-08-21,116.0
2195,B3SA3,B3SAV15,PUT,OTM,14.26,14.56,-2.060440,0.82,0.96,200,-0.335420,-0.003813,43.604,40.83,6.101190,2.542163,72,2026-10-16,143.5
1873,B3SA3,B3SAT144,PUT,OTM,14.26,14.56,-2.060440,0.51,0.52,31300,-0.353861,-0.007217,32.161,40.43,3.709091,3.272727,34,2026-08-21,153.5
1875,B3SA3,B3SAT150,PUT,ATM,14.46,14.56,-0.686813,0.59,0.61,53200,-0.393626,-0.007189,36.043,44.54,4.253785,3.753340,34,2026-08-21,157.5
1871,B3SA3,B3SAT141,PUT,OTM,14.01,14.56,-3.777473,0.42,0.43,14300,-0.305621,-0.007084,36.894,35.36,3.090508,2.726919,34,2026-08-21,168.0
7621,EGIE3,EGIET317,PUT,OTM,31.75,32.30,-1.702786,1.05,1.27,2000,-0.357047,-0.014211,35.181,40.35,3.420195,3.017820,34,2026-08-21,181.5
7623,EGIE3,EGIET322,PUT,ATM,32.25,32.30,-0.154799,1.29,1.54,2500,-0.405790,-0.014009,37.062,45.37,4.166667,3.676471,34,2026-08-21,185.5
7619,EGIE3,EGIET319,PUT,OTM,31.50,32.30,-2.476780,0.95,1.17,400,-0.333195,-0.014172,34.902,37.87,3.109656,2.743814,34,2026-08-21,187.0
13599,PETR4,PETRV437,PUT,OTM,36.80,37.89,-2.876748,1.69,1.75,400,-0.281265,-0.005631,35.117,33.07,4.813443,2.005601,72,2026-10-16,195.0


In [ ]:
calls = df_final[df_final["categoria"] == "CALL"].copy()

# Retorno apenas do prêmio
calls["retorno"] = (calls["bid"] / calls["preco_atual"]) * 100

# Retorno mensalizado
calls["retorno_mes"] = calls["retorno"] * (30 / calls["dias"].replace(0, 1))

# Distância do strike
calls["dist_strike"] = (
    (calls["strike"] / calls["preco_atual"]) - 1
) * 100


# -----------------------------
# FILTRO
# -----------------------------

filtro_call = calls[

    (calls["dias"] >= 15) &
    (calls["dist_strike"].between(0,np.inf)) &
    (calls["retorno_mes"] >= 0.70) &


].copy()

# -----------------------------
# SCORE
# -----------------------------

if not filtro_call.empty:

    filtro_call["score"] = (

        filtro_call["retorno_mes"].rank(ascending=False) * 4 +
        filtro_call["dist_strike"].rank(ascending=False) * 3 +
        filtro_call["volume"].rank(ascending=False) * 2 +
        filtro_call["theta"].rank(ascending=False) * 1

    )

    print("\n--- MELHORES OPÇÕES DE VENDA DE CALL ---")

    display(

        filtro_call[
            [
                "ativo", "symbol", "categoria", "moneyness", "strike",
                "preco_atual", "dist_strike", "bid", "ask", "volume", "delta",
                "theta", "vol", "poe", "retorno", "retorno_mes", "dias", "vencimento",
                "score",

            ]
        ]
        .sort_values("score")
        .head(30)

    )


--- MELHORES OPÇÕES DE VENDA DE CALL ---


,ativo,symbol,categoria,moneyness,strike,preco_atual,dist_strike,bid,ask,volume,delta,theta,vol,poe,retorno,retorno_mes,dias,vencimento,score
1884,B3SA3,B3SAH159,CALL,OTM,15.71,14.56,7.898352,0.42,0.43,23500,0.361150,-0.013481,34.980,31.28,2.884615,2.545249,34,2026-08-21,251.0
1888,B3SA3,B3SAH167,CALL,OTM,16.21,14.56,11.332418,0.29,0.31,11400,0.277037,-0.011710,36.720,23.45,1.991758,1.757434,34,2026-08-21,272.0
1882,B3SA3,B3SAH157,CALL,OTM,15.46,14.56,6.181319,0.50,0.51,24600,0.407333,-0.014217,34.925,35.68,3.434066,3.030058,34,2026-08-21,299.5
1886,B3SA3,B3SAH162,CALL,OTM,15.96,14.56,9.615385,0.35,0.37,5600,0.317589,-0.012636,35.003,27.19,2.403846,2.121041,34,2026-08-21,318.5
626,ABEV3,ABEVH177,CALL,OTM,17.00,15.87,7.120353,0.33,0.34,69400,0.335039,-0.011619,27.314,29.87,2.079395,1.834760,34,2026-08-21,333.0
3570,BBAS3,BBASH212,CALL,OTM,20.90,19.73,5.930056,0.44,0.46,53000,0.324924,-0.011568,25.742,29.80,2.230106,1.967741,34,2026-08-21,370.0
3572,BBAS3,BBASH215,CALL,OTM,21.15,19.73,7.197162,0.36,0.38,55500,0.270936,-0.010343,25.937,24.63,1.824633,1.609970,34,2026-08-21,380.0
1880,B3SA3,B3SAH154,CALL,OTM,15.21,14.56,4.464286,0.59,0.61,31600,0.455624,-0.014814,35.277,40.36,4.052198,3.575469,34,2026-08-21,381.0
21292,RADL3,RADLH180,CALL,OTM,17.93,16.78,6.853397,0.62,0.63,2100,0.402354,-0.017412,38.384,34.76,3.694875,3.260184,34,2026-08-21,383.0
622,ABEV3,ABEVH165,CALL,OTM,16.50,15.87,3.969754,0.49,0.51,141100,0.446906,-0.013307,27.253,40.69,3.087587,2.724341,34,2026-08-21,395.0
